# Gen in Pyodide

This site bundles `gen-0.3.1-cp311-abi3-pyemscripten_2025_0_wasm32.whl` in its site-relative piplite index. Install it
using the kernel's supported wheel mechanism. Gen runs in the kernel worker.


In [ ]:
import piplite
await piplite.install('gen==0.3.1')
import gen  # noqa: E402
print(gen.__version__)


## Create a repository and read a sequence

Use `/tmp` (the kernel's MEMFS) for live databases. JupyterLite's `/drive` uses
buffered Contents API files, which are unsuitable for live SQLite connections.
This repository lasts for this kernel session. Export it before restarting.


In [ ]:
from pathlib import Path
from tempfile import mkdtemp

root = Path(mkdtemp(prefix='gen-demo-', dir='/tmp'))
repository = gen.Repository(str(root))
fasta = root / 'small.fa'
fasta.write_text('>tiny\nACGTACGT\n')
repository.import_fasta(str(fasta), sample='initial')
repository.export_fasta(str(root / 'before.fa'), sample='initial')
print((root / 'before.fa').read_text())


## Edit and inspect committed history

Import and update methods commit their operation automatically. The region uses
zero-based half-open coordinates; replace the eight bases with a new sequence.


In [ ]:
repository.update_with_sequence('ACGTTTGT', 'initial', 'edited', 'tiny:0-8')
repository.export_fasta(str(root / 'after.fa'), sample='edited')
print((root / 'after.fa').read_text())
for operation in repository.get_operations():
    print(operation.id, operation.message)


## Browser HTTP example

The local demonstration server deliberately rejects a clone with HTTP 422.
The call goes through Gen's Rust/C XHR transport and preserves the server's
error body. This is an error-path demonstration, not a successful remote clone.
For a real clone, supply a GenHub-compatible URL whose server permits CORS.


In [ ]:
# Set this to the origin printed by serve.py if you use a different port.
remote_url = 'http://127.0.0.1:4502/api/repos/demo/rejected'
try:
    gen.clone(remote_url, str(root / 'remote-clone'))
except RuntimeError as error:
    print(error)


## Export the repository

Close all Gen objects before archiving, then copy the archive to `/drive` to
make it available in the file browser. Download it from there to keep it outside
browser storage. JupyterLite stores drive files in IndexedDB for this origin;
clearing site data deletes them. Installed wheels and `/tmp` are lost on restart.


In [ ]:
import gc
import shutil

# Release the SQLite handles before packaging the complete repository.
del repository
gc.collect()
archive = shutil.make_archive('/tmp/gen-demo', 'zip', root_dir=root)
destination = Path('/drive/gen-demo.zip')
shutil.copyfile(archive, destination)
print(f'Download {destination.name} from the JupyterLite file browser.')
